# Chapter 1 · Tenerife 2023 — Sentinel-2 severity + Sentinel-1 cross-check + FIRMS timeline
### (and the reusable template for all four fires)

**One recipe, applied to every fire:**
- **Sentinel-2** → burn severity (dNBR + RBR). *The measurement.*
- **Sentinel-1** → smoke-independent **cross-check** that the scar is real.
- **FIRMS (VIIRS/MODIS)** → **ignition point + spread path**, so the reader sees where it started and how it moved.
- **LLM descriptive layer → later, identical for all four.**

**2026 / La Mierla is the full showcase:** because it's active, all four run at once — FIRMS gives *live current hotspots*, Sentinel-1 gives extent *through the smoke*, Sentinel-2 gives severity once a clean scene lands. The finished fires (Tenerife 2023, Culebra 2022) are the same recipe without the "live" part.

This notebook measures **Tenerife 2023** (finished, in the archive, protected parks, official figure to validate against) and **is the template**: run the next fire by editing only the CONFIG cell.

| | |
|---|---|
| Fire | Arafo-Candelaria, NE Tenerife · ignition **15 Aug 2023** |
| Size | ~12,273 ha (Copernicus EMS EMSR685) |
| Protected | Teide National Park + Corona Forestal Natural Park |


## 0 · Setup

In [ ]:
# Colab: uncomment if needed
# !pip install -q earthengine-api geemap pandas requests

import ee, geemap, json, io, requests
import pandas as pd
import datetime as dt

ee.Authenticate()
ee.Initialize(project="ee-sanjusajimon76")
print("EE initialized")
print('Earth Engine initialised:', ee.__version__)

EE initialized
Earth Engine initialised: 1.7.35


## 1 · CONFIG — the ONLY cell that changes per fire

In [ ]:
CONFIG = {
    # --- identity ---
    'fire_id':          'tenerife_2023',
    'fire_name':        'Tenerife (Arafo-Candelaria)',
    'region':           'Canary Islands, Spain',
    'year':             2023,
    'ignition_date':    '2023-08-15',
    'containment_date': '2023-09-11',

    # --- area of interest [W, S, E, N] ---
    'aoi_bbox':   [-16.66, 28.26, -16.34, 28.56],

    # --- windows (tightened post-window to avoid autumn dry-down) ---
    'pre_start':  '2023-07-10', 'pre_end':  '2023-08-13',
    'post_start': '2023-09-12', 'post_end': '2023-10-05',

    # --- official reference (validation) ---
    'official_source':  'Copernicus EMS EMSR685',
    'official_area_ha': 12273,

    # --- FIRMS (ignition point + spread path) ---
    'firms_map_key': '38da194a340f838919a9d282c513b640',   # firms.modaps.eosdis.nasa.gov/api/map_key/
    'firms_source':  'VIIRS_SNPP_SP',       # archive for 2023; *_NRT for active/2026

    # ================= FIXED FOR ALL FIRES =================
    's2_collection':      'COPERNICUS/S2_SR_HARMONIZED',
    'cs_plus_collection': 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED',
    'cs_band':'cs_cdf', 'cs_threshold':0.60, 'scl_shadow_backup':True,
    'max_cloud_pct':70, 'scale_m':20, 'apply_offset':True,
    's1_pass':'DESCENDING', 's1_pol':'VH', 's1_confirm_db':0.3,
    'sev_thresholds': {'unburned_max':0.15,'low_max':0.27,
                       'moderate_low_max':0.44,'moderate_high_max':0.66},
}
AOI = ee.Geometry.Rectangle(CONFIG['aoi_bbox'])
T   = CONFIG['sev_thresholds']
print('AOI area (km2):', round(AOI.area(1).getInfo()/1e6, 1))

AOI area (km2): 1044.0


## 2 · Sentinel-2 → NBR (Cloud Score+ masked)

NBR = (B8 − B12)/(B8 + B12). Cloud Score+ per-pixel masking; anything unreadable stays absent, never guessed.

In [ ]:
AOI_BUFF = AOI.buffer(10000)   # composites reach past AOI so the offset control ring has data

def mask_s2_clouds(img):
    good = img.select(CONFIG['cs_band']).gte(CONFIG['cs_threshold'])
    if CONFIG['scl_shadow_backup']:
        good = good.And(img.select('SCL').neq(3))   # drop cloud shadow
    return img.updateMask(good)

def nbr(img):
    return img.normalizedDifference(['B8', 'B12']).rename('NBR')

def s2_nbr_composite(start, end):
    csplus = ee.ImageCollection(CONFIG['cs_plus_collection'])
    col = (ee.ImageCollection(CONFIG['s2_collection'])
           .filterBounds(AOI_BUFF).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CONFIG['max_cloud_pct']))
           .linkCollection(csplus, [CONFIG['cs_band']]).map(mask_s2_clouds))
    return col.map(nbr).select('NBR').median().clip(AOI_BUFF), col.size()

nbr_pre,  n_pre  = s2_nbr_composite(CONFIG['pre_start'],  CONFIG['pre_end'])
nbr_post, n_post = s2_nbr_composite(CONFIG['post_start'], CONFIG['post_end'])
print('Pre scenes:', n_pre.getInfo(), '| Post scenes:', n_post.getInfo())

Pre scenes: 6 | Post scenes: 5


## 3 · dNBR + RBR + severity (land-masked)

dNBR classified on USGS thresholds; RBR (Parks et al. 2014) reported as the cross-fire-comparable metric. Land mask (WorldCover ≠ water) keeps ocean out of an island burn.

In [ ]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(AOI_BUFF)
land_mask  = worldcover.neq(80)   # exclude permanent water (island: keep ocean out)

dnbr_raw = nbr_pre.subtract(nbr_post).rename('dNBR').updateMask(land_mask)

# offset: mean dNBR over an unburned LAND control ring (now inside the buffered composite)
offset_val = 0.0
if CONFIG['apply_offset']:
    control = AOI.buffer(8000).difference(AOI.buffer(3000))
    stats = dnbr_raw.reduceRegion(
        ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
        control, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).getInfo()
    offset_val = stats.get('dNBR_mean') or 0.0
    print('dNBR offset:', round(offset_val, 4), '| control pixels:', stats.get('dNBR_count'))

dnbr = dnbr_raw.subtract(offset_val).rename('dNBR')
rbr  = dnbr.divide(nbr_pre.add(1.001)).rename('RBR')

severity = (dnbr
    .where(dnbr.lt(T['unburned_max']), 0)
    .where(dnbr.gte(T['unburned_max']).And(dnbr.lt(T['low_max'])), 1)
    .where(dnbr.gte(T['low_max']).And(dnbr.lt(T['moderate_low_max'])), 2)
    .where(dnbr.gte(T['moderate_low_max']).And(dnbr.lt(T['moderate_high_max'])), 3)
    .where(dnbr.gte(T['moderate_high_max']), 4)
    .toInt().updateMask(land_mask).rename('severity'))
burn_mask = severity.gte(1)
print('dNBR / RBR / severity ready.')

dNBR offset: -0.0109 | control pixels: 880235
dNBR / RBR / severity ready.


## 4 · Severity area + normalized shares

In [6]:
SEV_NAMES = {1:'low',2:'moderate_low',3:'moderate_high',4:'high'}
grp = (ee.Image.pixelArea().divide(1e4).addBands(severity)
       .reduceRegion(ee.Reducer.sum().group(1,'class'),
                     AOI, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).getInfo())
severity_ha = {v:0.0 for v in SEV_NAMES.values()}
for g in grp.get('groups', []):
    if int(g['class']) in SEV_NAMES: severity_ha[SEV_NAMES[int(g['class'])]] = round(g['sum'],1)
total_burned_ha = round(sum(severity_ha.values()), 1)
severity_pct = {k:(round(100*v/total_burned_ha,1) if total_burned_ha else 0.0) for k,v in severity_ha.items()}
pct_high = severity_pct['high']
mean_rbr = round(ee.Number(rbr.updateMask(burn_mask).reduceRegion(
    ee.Reducer.mean(), AOI, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).get('RBR')).getInfo() or 0.0, 4)
print('Total burned (ha):', total_burned_ha, '| % high:', pct_high, '| mean RBR:', mean_rbr)

Total burned (ha): 13470.5 | % high: 13.5 | mean RBR: 0.2948


## 5 · Land-cover breakdown

In [ ]:
WC_NAMES = {10:'tree_cover',20:'shrubland',30:'grassland',40:'cropland',50:'built_up',
            60:'bare_sparse',70:'snow_ice',80:'water',90:'wetland',95:'mangrove',100:'moss_lichen'}
lcg = (ee.Image.pixelArea().divide(1e4).addBands(worldcover.updateMask(burn_mask))
       .reduceRegion(ee.Reducer.sum().group(1,'lc'),
                     AOI, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).getInfo())
landcover_burned_ha = {}
for g in lcg.get('groups', []):
    landcover_burned_ha[WC_NAMES.get(int(g['lc']), f"class_{int(g['lc'])}")] = round(g['sum'],1)
for key in ('tree_cover','shrubland','cropland'): landcover_burned_ha.setdefault(key, 0.0)
landcover_burned_pct = {k:(round(100*v/total_burned_ha,1) if total_burned_ha else 0.0)
                        for k,v in landcover_burned_ha.items()}
print('Burned by land cover (ha):', landcover_burned_ha)

Burned by land cover (ha): {'tree_cover': 8542.6, 'shrubland': 2197.3, 'grassland': 2432.8, 'cropland': 71.5, 'built_up': 182.2, 'bare_sparse': 44.2}


## 6 · Protected-area impact (WDPA / Natura 2000) — exposure, not damage

In [ ]:
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons').filterBounds(AOI)
protected_mask = ee.Image.constant(0).byte().paint(wdpa,1).gt(0).clip(AOI).updateMask(land_mask)
protected_area_burned_ha = round(ee.Number(
    ee.Image.pixelArea().divide(1e4).updateMask(burn_mask).updateMask(protected_mask)
      .reduceRegion(ee.Reducer.sum(), AOI, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True)
      .get('area')).getInfo() or 0.0, 1)
protected_share_pct = round(100*protected_area_burned_ha/total_burned_ha,1) if total_burned_ha else 0.0
protected_areas = wdpa.aggregate_array('NAME').distinct().getInfo()[:10]
print('Protected:', protected_areas, '|', protected_area_burned_ha, 'ha', f'({protected_share_pct}%)')

Protected: ['El Teide', 'Montaña de los Frailes', 'Barranco de Fasnia y Güímar', 'Las Lagunetas', 'Siete Lomas', 'La Resbala', 'Campeches, Tigaiga y Ruiz', 'Costa de Acentejo', 'Rambla de Castro', 'Corona Forestal'] | 10628.0 ha (78.9%)


## 7 · Validation vs the official figure

In [ ]:
official = CONFIG['official_area_ha']
pct_diff = round(100*(total_burned_ha-official)/official, 1)
print(f"Ours: {total_burned_ha} ha | {CONFIG['official_source']}: {official} ha | diff: {pct_diff:+.1f}%")

Ours: 13470.5 ha | Copernicus EMS EMSR685: 12273 ha | diff: +9.8%


## 8 · Sentinel-1 cross-check (smoke-independent)

Optical is blind through smoke; C-band SAR is not. A pre→post VH backscatter change over the scar, compared to the unburned surround, is an **independent confirmation** that the burn is real. Reported qualitatively (mean dB change inside vs outside) → `sar_confirms_scar`.

In [ ]:
def s1_mean(start, end):
    col = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filterBounds(AOI).filterDate(start, end)
           .filter(ee.Filter.eq('instrumentMode','IW'))
           .filter(ee.Filter.eq('orbitProperties_pass', CONFIG['s1_pass']))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', CONFIG['s1_pol']))
           .select(CONFIG['s1_pol']))
    return col.mean().clip(AOI), col.size()

s1_pre,  s1n_pre  = s1_mean(CONFIG['pre_start'],  CONFIG['pre_end'])
s1_post, s1n_post = s1_mean(CONFIG['post_start'], CONFIG['post_end'])
s1_change = s1_post.subtract(s1_pre).rename('dVH')

def _mean(mask):
    return ee.Number(s1_change.updateMask(mask).reduceRegion(
        ee.Reducer.mean(), AOI, 30, maxPixels=1e13, bestEffort=True).get('dVH')).getInfo()

sar_confirms_scar = None            # None = inconclusive / no data (shown as a gap, not a guess)
sar_delta_db = None
if s1n_pre.getInfo() > 0 and s1n_post.getInfo() > 0:
    cb, cu = _mean(burn_mask) or 0.0, _mean(burn_mask.Not()) or 0.0
    sar_delta_db = round(cb - cu, 3)
    if abs(sar_delta_db) >= CONFIG['s1_confirm_db']:
        sar_confirms_scar = True
    else:
        sar_confirms_scar = None     # too weak to confirm -> inconclusive, not a false negative
    print(f'VH change  scar: {cb:+.2f} dB | outside: {cu:+.2f} dB | Δ: {sar_delta_db:+.2f} dB'
          f' | confirms: {sar_confirms_scar}')
else:
    print('No S1 scene in a window — sar_confirms_scar = None (inconclusive)')

VH change  scar: +0.02 dB | outside: -0.18 dB | Δ: +0.20 dB | confirms: None


## 9 · FIRMS — ignition point + spread path (the reader's map)

VIIRS 375 m active-fire detections give the story the severity map can't: **where it started and how it spread.** Earliest detection = ignition point; detections ordered by time = the path. Exported as `firms_timeline_<id>.geojson` (each point carries its date) for the frontend to colour or animate.

Needs a free FIRMS **MAP_KEY** (set in CONFIG). For the **2026 active fire**, switch `firms_source` to a `*_NRT` product and this same cell shows **live current hotspots**.

In [ ]:
w,s,e,n = CONFIG['aoi_bbox']
area = f"{w},{s},{e},{n}"
d0 = dt.datetime.fromisoformat(CONFIG['ignition_date'])
d1 = dt.datetime.fromisoformat(CONFIG['containment_date'])

frames = []
cur = d0
while cur <= d1:
    span = min(5, (d1 - cur).days + 1)   # FIRMS area API allows 1..5 days per call
    url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
           f"{CONFIG['firms_map_key']}/{CONFIG['firms_source']}/{area}/{span}/{cur.date()}")
    try:
        r = requests.get(url, timeout=60)
        if r.ok and 'latitude' in r.text:
            frames.append(pd.read_csv(io.StringIO(r.text)))
        elif not r.ok:
            print('FIRMS', cur.date(), '->', r.status_code, r.text[:80])
    except Exception as ex:
        print('FIRMS fetch failed for', cur.date(), ':', ex)
    cur += dt.timedelta(days=span)

firms = {'ignition_lat':None,'ignition_lon':None,'first_detection':None,
         'last_detection':None,'active_days':None,'n_detections':0,
         'frp_max_mw':None,'frp_total_mw':None}

if frames:
    df = pd.concat(frames, ignore_index=True).drop_duplicates()
    df['ts'] = pd.to_datetime(df['acq_date'].astype(str) + ' ' +
                              df['acq_time'].astype(str).str.zfill(4).str[:2] + ':' +
                              df['acq_time'].astype(str).str.zfill(4).str[2:], errors='coerce')
    df = df.dropna(subset=['ts']).sort_values('ts').reset_index(drop=True)

    # which optional columns are present? (VIIRS: frp, bright_ti4 | MODIS: frp, brightness)
    conf_col   = 'confidence' if 'confidence' in df.columns else None
    frp_col    = 'frp' if 'frp' in df.columns else None
    bright_col = next((c for c in ('bright_ti4','brightness') if c in df.columns), None)
    print('columns available ->', 'confidence:', bool(conf_col),
          '| frp:', bool(frp_col), '| brightness:', bright_col)

    ig = df.iloc[0]
    firms = {
        'ignition_lat':   round(float(ig['latitude']), 5),
        'ignition_lon':   round(float(ig['longitude']), 5),
        'first_detection': str(df['ts'].min()),
        'last_detection':  str(df['ts'].max()),
        'active_days':     int((df['ts'].max() - df['ts'].min()).days) + 1,
        'n_detections':    int(len(df)),
        'frp_max_mw':      round(float(df[frp_col].max()), 1) if frp_col else None,
        'frp_total_mw':    round(float(df[frp_col].sum()), 1) if frp_col else None,
    }

    # export path GeoJSON — one point per detection, with date + intensity
    feats = []
    for r in df.itertuples(index=False):
        props = {'date': str(r.ts),
                 'confidence': str(getattr(r, conf_col)) if conf_col else ''}
        if frp_col:
            props['frp'] = float(getattr(r, frp_col) or 0)        # fire radiative power, MW
        if bright_col:
            props['bright_k'] = float(getattr(r, bright_col) or 0)  # brightness temp, Kelvin
        feats.append({
            'type': 'Feature',
            'geometry': {'type': 'Point',
                         'coordinates': [float(r.longitude), float(r.latitude)]},
            'properties': props})

    with open(f"firms_timeline_{CONFIG['fire_id']}.geojson", 'w') as f:
        json.dump({'type': 'FeatureCollection', 'features': feats}, f)

    print('FIRMS:', firms)
    print('ignition point:', firms['ignition_lat'], firms['ignition_lon'])
    if frp_col:
        print(f"FRP: max {firms['frp_max_mw']} MW · total {firms['frp_total_mw']} MW")
else:
    print('No FIRMS data (check MAP_KEY / source / dates). ignition point + path unavailable.')

columns available -> confidence: True | frp: True | brightness: bright_ti4
FIRMS: {'ignition_lat': 28.35905, 'ignition_lon': -16.43122, 'first_detection': '2023-08-16 02:23:00', 'last_detection': '2023-08-23 03:32:00', 'active_days': 8, 'n_detections': 1026, 'frp_max_mw': 318.6, 'frp_total_mw': 22703.8}
ignition point: 28.35905 -16.43122
FRP: max 318.6 MW · total 22703.8 MW


/tmp/ipykernel_2003/1602299282.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True).drop_duplicates()


## 10 · Visual QA — severity + ignition point + spread path

In [ ]:
sev_palette = ['00e400','ffff00','ffaa00','ff0000']
m = geemap.Map(); m.centerObject(AOI, 11)
m.addLayer(dnbr, {'min':-0.2,'max':1.0,'palette':['blue','white','red']}, 'dNBR', False)
m.addLayer(severity.updateMask(burn_mask), {'min':1,'max':4,'palette':sev_palette}, 'Severity')
m.addLayer(s1_change, {'min':-3,'max':3,'palette':['red','white','blue']}, 'S1 VH change', False)
m.addLayer(protected_mask.selfMask(), {'palette':['00b3ff']}, 'Protected', False)
if firms['ignition_lat'] is not None:
    ig_pt = ee.Geometry.Point([firms['ignition_lon'], firms['ignition_lat']])
    m.addLayer(ig_pt, {'color':'black'}, 'Ignition point')
m.addLayer(AOI, {'color':'gray'}, 'AOI', False)
m

Map(center=[28.410022680953727, -16.50000000000033], controls=(WidgetControl(options=['position', 'transparent…

## 11 · Export `facts.json` — locked comparison schema (now with S1 + FIRMS)

In [ ]:
facts = {
    'fire_id':CONFIG['fire_id'], 'fire_name':CONFIG['fire_name'], 'region':CONFIG['region'],
    'year':CONFIG['year'], 'ignition_date':CONFIG['ignition_date'],
    'containment_date':CONFIG['containment_date'],
    'sensors':'Sentinel-2 (severity), Sentinel-1 (cross-check), VIIRS/MODIS FIRMS (timeline)',
    'method':'dNBR (USGS/Key-Benson) + RBR (Parks et al. 2014); S1 VH change; FIRMS ignition+path',
    'pre_fire_window':[CONFIG['pre_start'],CONFIG['pre_end']],
    'post_fire_window':[CONFIG['post_start'],CONFIG['post_end']],
    'dnbr_offset_applied':round(offset_val,4),

    'total_burned_ha':total_burned_ha,
    'severity_ha':severity_ha, 'severity_pct':severity_pct, 'pct_high_severity':pct_high,
    'mean_rbr':mean_rbr,

    'landcover_burned_ha':landcover_burned_ha, 'landcover_burned_pct':landcover_burned_pct,
    'protected_area_burned_ha':protected_area_burned_ha,
    'protected_share_pct':protected_share_pct, 'protected_areas':protected_areas,

    'official_reference':{'source':CONFIG['official_source'],'area_ha':CONFIG['official_area_ha'],
                          'our_area_ha':total_burned_ha,'pct_diff':pct_diff},

    'sar_confirms_scar':sar_confirms_scar,
    'firms':firms,

    's2_scenes_pre':int(n_pre.getInfo()), 's2_scenes_post':int(n_post.getInfo()),
    'limitations':[
        'dNBR estimates severity, not a field-surveyed perimeter.',
        'Single post-fire composite; low-severity edges inflate area vs VHR delineation.',
        'Trade-wind cloud can leave gaps; masked pixels shown as absent, not guessed.',
        'S1 cross-check is qualitative confirmation, not an independent area estimate.',
        'FIRMS points are ~375 m detections; ignition point is first *detected* pixel, not exact origin.',
        'Protected-area figure is exposure (burn within boundaries), not ecological damage.',
    ],
    'generated_utc':dt.datetime.utcnow().isoformat()+'Z',
}
with open(f"facts_{CONFIG['fire_id']}.json",'w') as f:
    json.dump(facts, f, indent=2, ensure_ascii=False)
print(json.dumps(facts, indent=2, ensure_ascii=False))

{
  "fire_id": "tenerife_2023",
  "fire_name": "Tenerife (Arafo-Candelaria)",
  "region": "Canary Islands, Spain",
  "year": 2023,
  "ignition_date": "2023-08-15",
  "containment_date": "2023-09-11",
  "sensors": "Sentinel-2 (severity), Sentinel-1 (cross-check), VIIRS/MODIS FIRMS (timeline)",
  "method": "dNBR (USGS/Key-Benson) + RBR (Parks et al. 2014); S1 VH change; FIRMS ignition+path",
  "pre_fire_window": [
    "2023-07-10",
    "2023-08-13"
  ],
  "post_fire_window": [
    "2023-09-12",
    "2023-10-05"
  ],
  "dnbr_offset_applied": -0.0109,
  "total_burned_ha": 13470.5,
  "severity_ha": {
    "low": 4862.9,
    "moderate_low": 4047.4,
    "moderate_high": 2737.6,
    "high": 1822.6
  },
  "severity_pct": {
    "low": 36.1,
    "moderate_low": 30.0,
    "moderate_high": 20.3,
    "high": 13.5
  },
  "pct_high_severity": 13.5,
  "mean_rbr": 0.2948,
  "landcover_burned_ha": {
    "tree_cover": 8542.6,
    "shrubland": 2197.3,
    "grassland": 2432.8,
    "cropland": 71.5,
    "buil

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson

fc = {'type':'FeatureCollection','features':[
    {'type':'Feature',
     'geometry':{'type':'Point','coordinates':[float(r.longitude), float(r.latitude)]},
     'properties':{'time':str(r.ts),
                   'icon':'circle',
                   'iconstyle':{'fillColor':'#ff3300','fillOpacity':0.7,
                                'stroke':False,'radius':4}}}
    for r in df.itertuples(index=False)]}

cx, cy = df['longitude'].mean(), df['latitude'].mean()
fmap = folium.Map(location=[cy, cx], zoom_start=11, tiles='CartoDB positron')
folium.CircleMarker([firms['ignition_lat'], firms['ignition_lon']],
                    radius=7, color='black', fill=True, fill_opacity=1,
                    popup='Ignition').add_to(fmap)
TimestampedGeoJson(fc, period='PT12H', add_last_point=True,
                   auto_play=False, loop=False, transition_time=200).add_to(fmap)
fmap

In [ ]:
print(open(f"firms_timeline_{CONFIG['fire_id']}.geojson").read())

{"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": {"type": "Point", "coordinates": [-16.43122, 28.35905]}, "properties": {"date": "2023-08-16 02:23:00", "confidence": "n", "frp": 1.86, "bright_k": 304.73}}, {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-16.41881, 28.37541]}, "properties": {"date": "2023-08-16 02:23:00", "confidence": "n", "frp": 1.77, "bright_k": 306.67}}, {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-16.42278, 28.37583]}, "properties": {"date": "2023-08-16 02:23:00", "confidence": "n", "frp": 1.77, "bright_k": 314.6}}, {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-16.42658, 28.37623]}, "properties": {"date": "2023-08-16 02:23:00", "confidence": "n", "frp": 46.07, "bright_k": 341.83}}, {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-16.43041, 28.37664]}, "properties": {"date": "2023-08-16 02:23:00", "confidence": "h", "frp": 46.07, "bright_k": 367.0}}, {"type": "Fe

## Runbook — the other three fires (same recipe)

**Change only CONFIG.** Method, thresholds, schema stay fixed → comparable outputs.

- **Culebra 2022** → validate vs **EMSR580 / EMSR602**; two sub-fires (June + Losacio) — decide once: two events or one combined AOI. `firms_source` archive.
- **La Mierla 2026** → **all four satellites at once.** `firms_source` = `*_NRT` shows **live current hotspots**; S1 gives extent through the smoke now; run the S2 severity block once a clean post-fire scene exists. This is the full-picture event.
- **Valencia 2012** → **archive gap** (no Sentinel-2, no Landsat 5, only striped Landsat 7). Keep as the honest "where the archive runs out," or attempt with Landsat 7 and flag heavily. No S1 either (pre-2014).

**LLM descriptive layer — later, identical for all four.** Stack every `facts_*.json` + `firms_timeline_*.geojson` → `comparison.json` → one constrained model pass (same number-verification guardrail) → the comparative Chapter 2. The FIRMS ignition/path fields give the model the narrative spine (where it began, how far it ran, how much protected ground it crossed) without it inventing anything.

**Show the gaps** everywhere: unreadable Sentinel-2 pixels stay absent; a missing S1 window leaves `sar_confirms_scar = null`; missing FIRMS leaves the path empty — never guessed.


In [ ]:
# ============================================================
#  INTERACTIVE FIRE MAP  —  one cell, paste at the end of the notebook
#  Global satellite base · Sentinel-2 before/after swipe · FIRMS animation (FRP-scaled)
#  · burn-severity layer with its own legend
#  Needs: ee initialised, CONFIG, AOI, mask_s2_clouds,
#         firms_timeline_<id>.geojson + facts_<id>.json on disk
#  Optional: severity, burn_mask in memory (else the severity layer is skipped)
# ============================================================
import json, datetime as dt

# ---------- 1 · Sentinel-2 true-colour composites ----------
def s2_rgb(start, end):
    csplus = ee.ImageCollection(CONFIG['cs_plus_collection'])
    col = (ee.ImageCollection(CONFIG['s2_collection'])
           .filterBounds(AOI).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CONFIG['max_cloud_pct']))
           .linkCollection(csplus, [CONFIG['cs_band']])
           .map(mask_s2_clouds))
    return col.select(['B4','B3','B2']).median().clip(AOI)

RGB_VIS = {'min': 200, 'max': 3000, 'gamma': 1.15}
tile_url = lambda img, vis: img.getMapId(vis)['tile_fetcher'].url_format

url_pre  = tile_url(s2_rgb(CONFIG['pre_start'],  CONFIG['pre_end']),  RGB_VIS)
url_post = tile_url(s2_rgb(CONFIG['post_start'], CONFIG['post_end']), RGB_VIS)

SEV_COLORS = ['ffe08a', 'f9a03f', 'e8532b', '9d0208']
try:
    url_sev = tile_url(severity.updateMask(burn_mask),
                       {'min': 1, 'max': 4, 'palette': SEV_COLORS})
except Exception as e:
    url_sev = ''
    print('severity layer skipped:', e)

print('Sentinel-2 tiles ready.  NOTE: Earth Engine tile URLs expire after a few days —'
      ' export the composites as static tiles/COGs before publishing.')

# ---------- 2 · FIRMS detections -> compact string (lng,lat,FRP) ----------
gj = json.load(open(f"firms_timeline_{CONFIG['fire_id']}.geojson"))
by_ts, has_frp = {}, False
for f in gj['features']:
    lon, lat = f['geometry']['coordinates']
    pr  = f['properties']
    frp = float(pr.get('frp', 0) or 0)
    if frp: has_frp = True
    ts = pr['date'][:16].replace(' ', 'T')
    by_ts.setdefault(ts, []).append(
        f"{round((-lon-16)*1e5):05d},{round((lat-28)*1e5):05d},{round(frp)}")
RAW = ';'.join(f"{ts}|{' '.join(v)}" for ts, v in sorted(by_ts.items()))
n_det = sum(len(v) for v in by_ts.values())
print(f"{n_det} detections, {len(by_ts)} overpasses")
if not has_frp:
    print("WARNING: no FRP in the GeoJSON — re-run Cell 9 so properties include 'frp'."
          " Dots will be uniform size for now.")

# ---------- 3 · numbers from facts.json ----------
facts = json.load(open(f"facts_{CONFIG['fire_id']}.json"))
ign, b = facts['firms'], CONFIG['aoi_bbox']
sev_ha  = facts['severity_ha']
tot_ha  = facts['total_burned_ha']
SEV_ROWS = ''.join(
    f'<div class="key"><span class="dot" style="background:#{c}"></span>{n}'
    f'<b style="margin-left:auto;color:#fff">{sev_ha.get(k,0):,.0f} ha</b></div>'
    for k, n, c in [('low','Low','ffe08a'), ('moderate_low','Moderate-low','f9a03f'),
                    ('moderate_high','Moderate-high','e8532b'), ('high','High','9d0208')])

# ---------- 4 · page ----------
HTML = r'''<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>__NAME__ — how the fire spread</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@700&family=Space+Mono:wght@400;700&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css">
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<style>
:root{--paper:#0f1218;--ink:#f2f4f8;--sub:#9aa4b4;--line:#2a3240;--hot:#ef6a2b;--panel:rgba(16,20,27,.93)}
*{box-sizing:border-box}html,body{margin:0;height:100%}
body{background:var(--paper);color:var(--ink);font-family:Inter,system-ui,sans-serif;-webkit-font-smoothing:antialiased}
.fig{position:fixed;inset:0}#map{position:absolute;inset:0;background:#0b0e13}
#glow{position:absolute;inset:0;pointer-events:none;z-index:410}
#divider{position:absolute;top:0;bottom:0;width:2px;background:rgba(255,255,255,.85);
 box-shadow:0 0 8px rgba(0,0,0,.6);z-index:405;pointer-events:none;display:none}
#divider:after{content:'';position:absolute;top:50%;left:50%;width:30px;height:30px;margin:-15px 0 0 -15px;
 border-radius:50%;background:rgba(255,255,255,.9);box-shadow:0 0 10px rgba(0,0,0,.5)}
.head{position:absolute;top:0;left:262px;right:250px;z-index:500;padding:14px 18px 26px;pointer-events:none;
 background:linear-gradient(180deg,rgba(9,12,17,.9),rgba(9,12,17,.45) 60%,transparent)}
.eyebrow{font-family:"Space Mono",monospace;font-size:10px;letter-spacing:.2em;text-transform:uppercase;color:var(--hot);margin:0 0 5px}
h1{font-family:"Space Grotesk",sans-serif;font-size:clamp(17px,2.2vw,24px);line-height:1.12;margin:0 0 6px}
.dek{font-size:12.5px;line-height:1.5;color:#c3cad6;margin:0;max-width:62ch}.dek b{color:#fff}
.clock{position:absolute;top:12px;right:16px;z-index:520;text-align:right;background:var(--panel);
 border:1px solid var(--line);border-radius:11px;padding:9px 13px;pointer-events:none}
.clock .date{font-family:"Space Mono",monospace;font-weight:700;font-size:clamp(15px,2vw,21px)}
.clock .time{font-family:"Space Mono",monospace;font-size:11px;color:var(--sub)}
.clock .day{font-family:"Space Mono",monospace;font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--hot);margin-top:5px;font-weight:700}
.panel{position:absolute;z-index:500;background:var(--panel);border:1px solid var(--line);border-radius:11px;padding:12px 13px}
.panel h2{font-family:"Space Mono",monospace;font-size:9px;letter-spacing:.16em;text-transform:uppercase;color:var(--sub);margin:0 0 9px;font-weight:700}
.layers{left:16px;top:14px;width:232px}
.lay{display:flex;align-items:center;gap:8px;font-size:12px;margin:5px 0;cursor:pointer;color:#c3cad6}
.lay input{accent-color:var(--hot);flex:none}
.sec{margin-top:9px;padding-top:9px;border-top:1px solid var(--line)}
.sec label.rng{font-family:"Space Mono",monospace;font-size:9.5px;letter-spacing:.08em;text-transform:uppercase;color:var(--sub);display:flex;justify-content:space-between;margin-bottom:4px}
.hint{font-size:10px;line-height:1.4;color:var(--sub);margin:5px 0 0}
.legend,.sevleg{left:16px;bottom:148px;width:232px}
.ramp{height:9px;border-radius:5px;background:linear-gradient(90deg,#f9a03f,#ef6a2b,#d62828,#6a040f);margin-bottom:4px}
.rl{display:flex;justify-content:space-between;font-family:"Space Mono",monospace;font-size:10px;color:var(--sub)}
.key{display:flex;align-items:center;gap:8px;font-size:11.5px;color:#c3cad6;margin-top:7px}
.dot{width:11px;height:11px;border-radius:50%;flex:none}
.dot.ign{background:#fff;border:3px solid #000}.dot.det{background:var(--hot)}
.sizes{display:flex;align-items:flex-end;gap:9px;margin-top:9px}
.sizes i{display:block;border-radius:50%;background:var(--hot);opacity:.85}
.note{margin-top:10px;padding-top:9px;border-top:1px solid var(--line);font-size:10px;line-height:1.45;color:var(--sub)}
.loc{right:16px;bottom:148px;width:180px}#mini{height:104px;border-radius:7px;overflow:hidden;border:1px solid var(--line)}
.loc p{margin:6px 0 0;font-size:10px;line-height:1.4;color:var(--sub)}
.ctl{position:absolute;left:0;right:0;bottom:0;z-index:510;display:flex;align-items:center;gap:12px;padding:13px 18px 15px;
 background:linear-gradient(0deg,rgba(9,12,17,.95),rgba(9,12,17,.6) 65%,transparent)}
button.play{flex:none;width:43px;height:43px;border-radius:50%;border:none;cursor:pointer;background:var(--hot);color:#140800;
 display:grid;place-items:center;box-shadow:0 0 18px rgba(239,106,43,.5)}
button.play svg{width:19px;height:19px}
.ghost{flex:none;background:transparent;border:1px solid var(--line);color:var(--sub);height:32px;padding:0 11px;
 border-radius:8px;cursor:pointer;font-family:"Space Mono",monospace;font-size:11px}
.ghost:hover{color:#fff;border-color:#48566c}
.track{flex:1;min-width:70px;display:flex;align-items:center}
input[type=range]{width:100%;-webkit-appearance:none;height:5px;border-radius:4px;background:#2b3442;outline:none;cursor:pointer}
input[type=range]::-webkit-slider-thumb{-webkit-appearance:none;width:15px;height:15px;border-radius:50%;background:#fff;border:2px solid var(--hot)}
input[type=range]::-moz-range-thumb{width:15px;height:15px;border-radius:50%;background:#fff;border:2px solid var(--hot)}
.count{flex:none;font-family:"Space Mono",monospace;font-size:11px;color:var(--sub);min-width:176px;text-align:right}
.count b{color:#fff}
.leaflet-control-scale{margin-bottom:68px!important;margin-left:258px!important}
.leaflet-control-zoom{margin-bottom:148px!important;margin-right:210px!important}
.leaflet-control-scale-line{background:rgba(0,0,0,.5)!important;color:#fff!important;border-color:#7d8798!important}
@media(max-width:1080px){.legend,.sevleg,.loc{display:none}.head{left:262px;right:16px}.leaflet-control-scale{margin-left:12px!important}}
@media(max-width:760px){.layers{display:none}.head{left:0;right:0}.count{display:none}.clock{top:auto;bottom:118px;right:12px}}
</style>
<div class="fig">
 <div id="map"></div><canvas id="glow"></canvas><div id="divider"></div>

 <div class="head"><p class="eyebrow">Groundtruth · Satellite journalism</p>
  <h1>How the __NAME__ fire spread, day by day</h1>
  <p class="dek">Each mark is where a satellite detected active fire; bigger marks burned more fiercely. It began on <b>__IGNDATE__</b> and burned for __NDAYS__ days.</p></div>

 <div class="clock"><div class="date" id="cDate">—</div><div class="time" id="cTime">&nbsp;</div><div class="day" id="cDay">&nbsp;</div></div>

 <div class="panel layers"><h2>What you're looking at</h2>
  <label class="lay"><input type="radio" name="view" value="base"> Global satellite only</label>
  <label class="lay"><input type="radio" name="view" value="pre"> + Sentinel-2 before the fire</label>
  <label class="lay"><input type="radio" name="view" value="post"> + Sentinel-2 after the fire</label>
  <label class="lay"><input type="radio" name="view" value="swipe" checked> + Compare before / after</label>
  <div class="sec" id="swipeWrap"><label class="rng"><span>&#9664; Before</span><span>After &#9654;</span></label>
   <input type="range" id="swipe" min="0" max="100" value="50">
   <p class="hint">Drag to wipe between the two Sentinel-2 images.</p></div>
  <div class="sec"><label class="rng"><span>Image opacity</span><span id="opVal">100%</span></label>
   <input type="range" id="opacity" min="0" max="100" value="100">
   <p class="hint">Fades Sentinel-2 to reveal the global satellite base underneath.</p></div>
  <div class="sec">
   <label class="lay"><input type="checkbox" id="lFire" checked> Fire detections (animated)</label>
   <label class="lay" id="sevRow"><input type="checkbox" id="lSev"> Burn severity (final)</label>
   <label class="lay"><input type="checkbox" id="lStreet" checked> Place names</label>
  </div>
 </div>

 <div class="panel legend" id="fireLeg"><h2>Fire detections</h2>
  <div class="ramp"></div>
  <div class="rl"><span>__D0__</span><span>__D1__</span></div>
  <div class="sizes">
   <span style="text-align:center"><i style="width:7px;height:7px;margin:0 auto 4px"></i>
    <span style="font-family:'Space Mono',monospace;font-size:9px;color:var(--sub)">low</span></span>
   <span style="text-align:center"><i style="width:12px;height:12px;margin:0 auto 4px"></i>
    <span style="font-family:'Space Mono',monospace;font-size:9px;color:var(--sub)">__FRPMID__</span></span>
   <span style="text-align:center"><i style="width:18px;height:18px;margin:0 auto 4px"></i>
    <span style="font-family:'Space Mono',monospace;font-size:9px;color:var(--sub)">__FRPMAX__ MW</span></span>
   <span style="font-size:10px;color:var(--sub);padding-bottom:2px">fire power</span>
  </div>
  <div class="key"><span class="dot ign"></span> Where the fire started</div>
  <p class="note">Colour = the day it burned. Size = fire radiative power, how much heat the satellite measured.</p></div>

 <div class="panel sevleg" id="sevLeg" style="display:none"><h2>Burn severity · dNBR</h2>
  __SEVROWS__
  <p class="note">Total __TOTHA__ ha, measured once from Sentinel-2 after the smoke cleared — a final result, not a daily sequence.</p></div>

 <div class="panel loc"><h2>Where this is</h2><div id="mini"></div><p>__REGION__</p></div>

 <div class="ctl">
  <button class="play" id="play" aria-label="Play"><svg id="pIcon" viewBox="0 0 24 24" fill="currentColor"><path d="M8 5v14l11-7z"/></svg></button>
  <button class="ghost" id="restart">Restart</button><button class="ghost" id="speed">1&times;</button>
  <button class="ghost" id="fit">Reset view</button>
  <div class="track"><input id="scrub" type="range" min="0" max="1000" value="0" aria-label="Timeline"></div>
  <div class="count"><b id="cN">0</b> detections &middot; <b id="cKm">0</b> km from start</div></div>
</div>
<script>
var RAW="__RAW__", IGN={lng:__IGNLON__,lat:__IGNLAT__};
var U_PRE="__UPRE__",U_POST="__UPOST__",U_SEV="__USEV__";
var BBOX=[__W__,__S__,__E__,__N__];

var PTS=[];
RAW.split(';').forEach(function(g){var p=g.split('|'),t=Date.parse(p[0]+':00Z');
 p[1].trim().split(' ').forEach(function(q){var a=q.split(',');
  PTS.push({lng:-16-(+a[0])/1e5,lat:28+(+a[1])/1e5,t:t,frp:+(a[2]||0)});});});
PTS.sort(function(a,b){return a.t-b.t;});
var tMin=PTS[0].t,tMax=PTS[PTS.length-1].t,span=tMax-tMin,DAY=864e5,nDays=Math.ceil(span/DAY)+1;
var FRPMAX=PTS.reduce(function(m,p){return Math.max(m,p.frp);},0)||1;
var RAMP=['#f9a03f','#ef6a2b','#d62828','#9d0208','#6a040f'];
function hx(h){return [parseInt(h.slice(1,3),16),parseInt(h.slice(3,5),16),parseInt(h.slice(5,7),16)];}
PTS.forEach(function(p){var f=(p.t-tMin)/span,s=f*(RAMP.length-1),
 i=Math.min(RAMP.length-2,Math.floor(s)),k=s-i,A=hx(RAMP[i]),B=hx(RAMP[i+1]);
 p.c=A.map(function(v,j){return Math.round(v+(B[j]-v)*k);});
 p.sz=5+9*Math.sqrt(Math.min(1,p.frp/FRPMAX));});

var map=L.map('map',{zoomControl:true,attributionControl:false});
var baseSat=L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',{maxZoom:19}).addTo(map);
map.createPane('pPre');map.getPane('pPre').style.zIndex=250;
map.createPane('pPost');map.getPane('pPost').style.zIndex=260;
map.createPane('pSev');map.getPane('pSev').style.zIndex=270;
map.createPane('pLab');map.getPane('pLab').style.zIndex=340;map.getPane('pLab').style.pointerEvents='none';
var labels=L.tileLayer('https://{s}.basemap.cartocdn.com/rastertiles/voyager_only_labels/{z}/{x}/{y}{r}.png',{maxZoom:19,subdomains:'abcd',pane:'pLab'}).addTo(map);
L.control.scale({imperial:false,position:'bottomleft'}).addTo(map);
var lyPre=L.tileLayer(U_PRE,{pane:'pPre',maxZoom:19});
var lyPost=L.tileLayer(U_POST,{pane:'pPost',maxZoom:19});
var lySev=U_SEV?L.tileLayer(U_SEV,{pane:'pSev',maxZoom:19,opacity:.85}):null;
if(!lySev){document.getElementById('sevRow').style.display='none';}

var bounds=L.latLngBounds([[BBOX[1],BBOX[0]],[BBOX[3],BBOX[2]]]);
function fitAll(){map.fitBounds(bounds,{padding:[40,40]});}fitAll();
L.marker([IGN.lat,IGN.lng],{zIndexOffset:900,icon:L.divIcon({className:'',iconSize:[20,20],iconAnchor:[10,10],
 html:'<div style="width:13px;height:13px;border-radius:50%;background:#fff;border:3px solid #000;box-shadow:0 0 0 3px rgba(0,0,0,.5)"></div>'})})
 .addTo(map).bindTooltip('Fire started here',{direction:'top',offset:[0,-10]});

var mini=L.map('mini',{zoomControl:false,attributionControl:false,dragging:false,scrollWheelZoom:false,
 doubleClickZoom:false,boxZoom:false,keyboard:false,touchZoom:false}).setView([IGN.lat+3,IGN.lng+3],4);
L.tileLayer('https://{s}.basemap.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png',{subdomains:'abcd'}).addTo(mini);
L.circleMarker([IGN.lat,IGN.lng],{radius:6,color:'#ef6a2b',weight:2,fillColor:'#ef6a2b',fillOpacity:.9}).addTo(mini);
setTimeout(function(){mini.invalidateSize();},400);

/* ---- swipe: clip the tile CONTAINER in layer-point space ---- */
var divider=document.getElementById('divider');
function currentMode(){return document.querySelector('input[name=view]:checked').value;}
function applySwipe(){
  var c=lyPost.getContainer();
  if(!c||!map.hasLayer(lyPost)){divider.style.display='none';return;}
  var v=+document.getElementById('swipe').value, size=map.getSize();
  var nw=map.containerPointToLayerPoint([0,0]), se=map.containerPointToLayerPoint([size.x,size.y]);
  var clipX=nw.x+(se.x-nw.x)*v/100;
  c.style.clip='rect('+[nw.y,se.x,se.y,clipX].join('px,')+'px)';
  divider.style.display='block';
  divider.style.left=(size.x*v/100)+'px';
}
function clearClip(){
  [lyPre,lyPost].forEach(function(l){var c=l.getContainer();
    if(c){c.style.clip='';c.style.clipPath='';c.style.webkitClipPath='';}});
  ['pPre','pPost'].forEach(function(n){var p=map.getPane(n);
    p.style.clipPath='none';p.style.webkitClipPath='none';});
  divider.style.display='none';
}
function setView(mode){
  if(map.hasLayer(lyPre))map.removeLayer(lyPre);
  if(map.hasLayer(lyPost))map.removeLayer(lyPost);
  clearClip();
  document.getElementById('swipeWrap').style.display=(mode==='swipe')?'block':'none';
  if(mode==='pre'){lyPre.addTo(map);}
  else if(mode==='post'){lyPost.addTo(map);}
  else if(mode==='swipe'){lyPre.addTo(map);lyPost.addTo(map);setTimeout(applySwipe,60);}
}
document.querySelectorAll('input[name=view]').forEach(function(r){
  r.onchange=function(){if(r.checked)setView(r.value);};});
document.getElementById('swipe').oninput=applySwipe;
map.on('move zoom zoomend moveend resize load',function(){
  if(currentMode()==='swipe')applySwipe();});
lyPost.on('load',function(){if(currentMode()==='swipe')applySwipe();});

document.getElementById('opacity').oninput=function(e){var o=e.target.value/100;
 lyPre.setOpacity(o);lyPost.setOpacity(o);
 document.getElementById('opVal').textContent=e.target.value+'%';};
var showFire=true;
document.getElementById('lFire').onchange=function(e){
 showFire=e.target.checked;
 document.getElementById('fireLeg').style.display=showFire?'block':'none';draw();};
document.getElementById('lSev').onchange=function(e){
 if(!lySev)return;
 e.target.checked?lySev.addTo(map):map.removeLayer(lySev);
 document.getElementById('sevLeg').style.display=e.target.checked?'block':'none';
 document.getElementById('fireLeg').style.display=(e.target.checked||!showFire)?'none':'block';};
document.getElementById('lStreet').onchange=function(e){
 e.target.checked?labels.addTo(map):map.removeLayer(labels);};
setView('swipe');

/* ---- fire glow ---- */
var cv=document.getElementById('glow'),ctx=cv.getContext('2d');
function rs(){var s=map.getSize(),d=Math.min(2,window.devicePixelRatio||1);
 cv.width=s.x*d;cv.height=s.y*d;cv.style.width=s.x+'px';cv.style.height=s.y+'px';ctx.setTransform(d,0,0,d,0,0);}
rs();map.on('resize',rs);
var FL=8*36e5;
function draw(){
 var s=map.getSize();ctx.clearRect(0,0,s.x,s.y);
 if(!showFire)return;
 ctx.globalCompositeOperation='lighter';
 for(var i=0;i<PTS.length;i++){var p=PTS[i];if(p.t>clock)continue;
  var q=map.latLngToContainerPoint([p.lat,p.lng]);
  if(q.x<-30||q.y<-30||q.x>s.x+30||q.y>s.y+30)continue;
  var f=Math.max(0,1-(clock-p.t)/FL),r=p.c[0],g=p.c[1],b=p.c[2],rad=p.sz+f*9;
  var gr=ctx.createRadialGradient(q.x,q.y,0,q.x,q.y,rad);
  gr.addColorStop(0,'rgba('+r+','+g+','+b+','+(.5+f*.35)+')');
  gr.addColorStop(.55,'rgba('+r+','+g+','+b+',.22)');
  gr.addColorStop(1,'rgba('+r+','+g+','+b+',0)');
  ctx.fillStyle=gr;ctx.beginPath();ctx.arc(q.x,q.y,rad,0,6.2832);ctx.fill();
  if(f>0){ctx.fillStyle='rgba(255,'+(210+Math.round(f*40))+',150,'+(f*.9)+')';
   ctx.beginPath();ctx.arc(q.x,q.y,2+f*2.5,0,6.2832);ctx.fill();}}
 ctx.globalCompositeOperation='source-over';}

/* ---- clock, timeline, controls ---- */
var MON=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'];
function fD(d){return d.getUTCDate()+' '+MON[d.getUTCMonth()]+' '+d.getUTCFullYear();}
function fT(d){return ('0'+d.getUTCHours()).slice(-2)+':'+('0'+d.getUTCMinutes()).slice(-2)+' UTC';}
function km(a,b){var R=6371,x=(b.lat-a.lat)*Math.PI/180,y=(b.lng-a.lng)*Math.PI/180,
 h=Math.pow(Math.sin(x/2),2)+Math.cos(a.lat*Math.PI/180)*Math.cos(b.lat*Math.PI/180)*Math.pow(Math.sin(y/2),2);
 return R*2*Math.atan2(Math.sqrt(h),Math.sqrt(1-h));}
var red=window.matchMedia('(prefers-reduced-motion: reduce)').matches;
var clock=red?tMax:tMin,playing=false,sp=0,SP=[1,2,4],$=function(i){return document.getElementById(i);};

function ui(){var d=new Date(clock);$('cDate').textContent=fD(d);$('cTime').textContent=fT(d);
 $('cDay').textContent='Day '+Math.min(nDays,Math.floor((clock-tMin)/DAY)+1)+' of '+nDays;
 var n=0,far=0;for(var i=0;i<PTS.length;i++){if(PTS[i].t<=clock){n++;var k=km(IGN,PTS[i]);if(k>far)far=k;}}
 $('cN').textContent=n.toLocaleString();$('cKm').textContent=far.toFixed(1);
 $('scrub').value=Math.round((clock-tMin)/span*1000);}

function setPlay(v){playing=v;
 $('pIcon').innerHTML=v?'<path d="M6 5h4v14H6zM14 5h4v14h-4z"/>':'<path d="M8 5v14l11-7z"/>';
 $('play').setAttribute('aria-label',v?'Pause':'Play');}

/* playback implies you want to see the fire — switch detections back on */
function ensureFireOn(){
  if(showFire)return;
  showFire=true;
  $('lFire').checked=true;
  $('fireLeg').style.display=$('lSev').checked?'none':'block';
  draw();
}

var last=performance.now();
function frame(now){var dt=now-last;last=now;
 if(playing){clock+=dt*(span/16000)*SP[sp];if(clock>=tMax){clock=tMax;setPlay(false);}ui();}
 draw();requestAnimationFrame(frame);}
ui();requestAnimationFrame(frame);
if(!red)setTimeout(function(){setPlay(true);},700);

$('play').onclick=function(){ensureFireOn();if(clock>=tMax)clock=tMin;setPlay(!playing);};
$('restart').onclick=function(){ensureFireOn();clock=tMin;ui();setPlay(true);};
$('speed').onclick=function(e){sp=(sp+1)%SP.length;e.target.innerHTML=SP[sp]+'&times;';};
$('fit').onclick=fitAll;
$('scrub').oninput=function(e){ensureFireOn();setPlay(false);clock=tMin+(e.target.value/1000)*span;ui();};
map.on('move zoom',function(){if(!playing)draw();});
</script>'''

frp_vals = [int(p.split(',')[2]) for v in by_ts.values() for p in v]
frp_max  = max(frp_vals) if frp_vals else 0
d0 = dt.datetime.fromisoformat(ign['first_detection'])
d1 = dt.datetime.fromisoformat(ign['last_detection'])
rep = {
 '__NAME__': facts['fire_name'], '__REGION__': facts['region'],
 '__IGNDATE__': d0.strftime('%d %B %Y'), '__NDAYS__': str(ign['active_days']),
 '__D0__': d0.strftime('%d %b'), '__D1__': d1.strftime('%d %b'),
 '__RAW__': RAW, '__IGNLON__': str(ign['ignition_lon']), '__IGNLAT__': str(ign['ignition_lat']),
 '__UPRE__': url_pre, '__UPOST__': url_post, '__USEV__': url_sev,
 '__W__': str(b[0]), '__S__': str(b[1]), '__E__': str(b[2]), '__N__': str(b[3]),
 '__SEVROWS__': SEV_ROWS, '__TOTHA__': f"{tot_ha:,.0f}",
 '__FRPMID__': str(round(frp_max/4)), '__FRPMAX__': str(frp_max),
}
for k, v in rep.items():
    HTML = HTML.replace(k, v)

out = f"fire_spread_{CONFIG['fire_id']}.html"
open(out, 'w').write(HTML)
print('wrote', out, f'({len(HTML)/1024:.0f} KB)')

from IPython.display import IFrame, display
display(IFrame(out, width='100%', height=660))

Sentinel-2 tiles ready.  NOTE: Earth Engine tile URLs expire after a few days — export the composites as static tiles/COGs before publishing.
1026 detections, 18 overpasses
wrote fire_spread_tenerife_2023.html (34 KB)
